In [ ]:
import numpy as np
import cv2
from skimage.exposure import equalize_adapthist
from PIL import Image
from pathlib import Path

In [ ]:
DATA_DIR = Path.cwd() / "data"
ORIGINAL_DIR = DATA_DIR / "original"
LABELS_DIR = DATA_DIR / "labels"

In [ ]:
def list_image_pairs():
    originals = sorted(ORIGINAL_DIR.glob("*.ppm"))
    labels = sorted(LABELS_DIR.glob("*.vk.ppm"))
    pairs = []
    for orig in originals:
        lbl = LABELS_DIR / f"{orig.stem}.vk.ppm"
        if lbl.exists():
            pairs.append((orig, lbl))
    return pairs


pairs = list_image_pairs()
print(f"Found {len(pairs)} input-target pairs")
for inp, tgt in pairs[:3]:
    print(f"  {inp.name} -> {tgt.name}")

In [ ]:
def extract_rgb_channels(image_path: Path):
    img = np.array(Image.open(image_path))
    return img[:, :, 0], img[:, :, 1], img[:, :, 2]

In [ ]:
def create_eye_mask(image_path: Path):
    img = cv2.imread(str(image_path))
    mask = np.zeros(img.shape[:2], np.uint8)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    h, w = img.shape[:2]
    margin = 5
    rect = (margin, margin, w - 2 * margin, h - 2 * margin)
    cv2.grabCut(img, mask, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
    result = np.where((mask == cv2.GC_FGD) | (mask == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
    return result > 0

In [ ]:
import matplotlib.pyplot as plt

def show_images(images, titles=None, cols = None, figsize_per_image=4, cmap="gray"):
    n = len(images)
    if cols is None:
        cols = n

    rows = max(1, (n + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_image, rows * figsize_per_image))
    axes = np.atleast_1d(axes).ravel()
    for i, src in enumerate(images):
        if isinstance(src, (str, Path)):
            img = np.array(Image.open(src))
        else:
            img = src
        axes[i].imshow(img, cmap=cmap)
        if titles and i < len(titles):
            axes[i].set_title(titles[i])
        axes[i].axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
for orig, targ in pairs[:2]:
    r, g, b = extract_rgb_channels(orig)
    mask = create_eye_mask(orig)
    show_images([orig, targ, r, g, mask])

In [ ]:
def apply_clahe(image, mask=None, clip_limit=0.03, nbins=256):
    if isinstance(image, (str, Path)):
        img = np.array(Image.open(image)).astype(float) / 255.0
    else:
        img = image.astype(float) / 255.0
    if mask is not None:
        img[~mask] = 0
    enhanced = equalize_adapthist(img, kernel_size=None, clip_limit=clip_limit, nbins=nbins)
    if mask is not None:
        enhanced[~mask] = 0
    return enhanced


def preprocess_image(image, mask, *, gauss: tuple[int, int] = (13, 13)):
    if isinstance(image, (str, Path)):
        img = np.array(Image.open(image)).astype(np.uint8) / 255.0
    else:
        img = image.astype(np.uint8)

    bg_map = cv2.GaussianBlur(img, gauss, 0)
    flat_bg = cv2.subtract(bg_map, img)
    flat_bg = np.clip(flat_bg, 0, 255).astype(np.uint8)

    return apply_clahe(flat_bg, mask=mask)


mask = create_eye_mask(pairs[0][0])
r, g, b = extract_rgb_channels(pairs[0][0])
r_res = preprocess_image(r, mask)
g_res = preprocess_image(g, mask)
b_res = preprocess_image(b, mask)

show_images([pairs[0][0], r, g, b], titles=['Green', 'Green CLAHE'], cmap='gray')
show_images([pairs[0][1], r_res, g_res, b_res], titles=['Green', 'Green CLAHE'], cmap='gray')


In [ ]:
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import StandardScaler


def extract_patches(image, mask, patch_size, stride):
    half = patch_size // 2
    h, w = image.shape
    patches = []
    coords = []
    for y in range(half, h - half, stride):
        for x in range(half, w - half, stride):
            if not mask[y, x]:
                continue
            patch = image[y - half:y + half + 1, x - half:x + half + 1]
            patches.append(patch)
            coords.append((y, x))
    return patches, coords


def patch_features(patch):
    feats = []
    feats.append(float(np.mean(patch)))
    feats.append(float(np.std(patch)))
    feats.append(float(np.min(patch)))
    feats.append(float(np.max(patch)))
    for p in [5, 10, 25, 50, 75, 90, 95]:
        feats.append(float(np.percentile(patch, p)))
    gy, gx = np.gradient(patch.astype(float))
    gm = np.sqrt(gy ** 2 + gx ** 2)
    feats.append(float(np.mean(gm)))
    feats.append(float(np.std(gm)))
    return np.array(feats, dtype=np.float32)


In [ ]:
def build_training_dataset(pairs, patch_size=32, stride=16, clip_limit=0.03, verbose=True):
    X_all = []
    y_all = []
    for inp_path, lbl_path in pairs:
        img = np.array(Image.open(inp_path))
        gold = np.array(Image.open(lbl_path))
        gold_bin = (gold > 0).astype(np.uint8)

        mask = create_eye_mask(inp_path)
        r, g, b = extract_rgb_channels(inp_path)
        data = preprocess_image(g, mask)
        data = (data * 255).astype(np.uint8)

        patches, coords = extract_patches(data, mask, patch_size, stride)
        for p, (yy, xx) in zip(patches, coords):
            X_all.append(p.flatten())
            y_all.append(int(gold_bin[yy, xx]))
        if verbose:
            n_vein = sum(1 for y in y_all[-len(patches):] if y == 1)
            print(f"{inp_path.stem}: {len(patches)} patches, {n_vein} vein")
    return np.array(X_all, dtype=np.float32), np.array(y_all, dtype=np.int32)


X, y = build_training_dataset(pairs[:1], patch_size=5, stride=1)
print(X[:10])
print(y[:10])
print(f"\nTotal: X={X.shape}, y={y.shape}, veins={y.sum()}, BG={(y == 0).sum()}")


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score


def make_classifier_pipeline(model=None):
    if model is None:
        model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    return Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', model),
    ])


X, y = build_training_dataset(pairs[:2], patch_size=32, stride=16)
print(f'Dataset: X={X.shape}, y={y.shape}')
print(f'Vein patches: {y.sum()}, BG patches: {(y == 0).sum()}')

clf = make_classifier_pipeline()
scores = cross_val_score(clf, X, y, cv=3, scoring='f1', n_jobs=-1)
print(f'Cross-val F1: {scores.mean():.3f} +/- {scores.std():.3f}')


In [ ]:
from sklearn.pipeline import Pipeline
# from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, balanced_accuracy_score, precision_score, recall_score


def make_balanced_pipeline(model=None):
    if model is None:
        model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    return Pipeline([
        ('scaler', StandardScaler()),
        # ('smote', SMOTE(random_state=42, k_neighbors=3)),
        ('classifier', model),
    ])


X, y = build_training_dataset(pairs[:2], patch_size=5, stride=1)
print(f'Dataset: X={X.shape}, y={y.shape}')
print(f'Vein patches: {y.sum()}, BG patches: {(y == 0).sum()}')
print(f'Imbalance ratio: {y.sum() / max(1, (y == 0).sum()):.3f}')

clf = make_balanced_pipeline()
scoring = {
    'f1': 'f1',
    'balanced_acc': make_scorer(balanced_accuracy_score),
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scores = cross_validate(clf, X, y, cv=cv, scoring=scoring, n_jobs=-1)
for metric in scoring:
    vals = scores[f'test_{metric}']
    print(f'{metric:>12}: {vals.mean():.3f} +/- {vals.std():.3f}')


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


X, y = build_training_dataset(pairs[:10], patch_size=32, stride=16, verbose=False)
print(f'Dataset: X={X.shape}, y={y.shape}')
print(f'Vein: {y.sum()}, BG: {(y == 0).sum()}, Ratio: {y.sum() / max(1, (y == 0).sum()):.3f}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

clf = make_balanced_pipeline()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
sensitivity = tp / max(1, tp + fn)
specificity = tn / max(1, tn + fp)
balanced_acc = (sensitivity + specificity) / 2

print(f'\n{"="*40}')
print(f'Accuracy:      {accuracy:.4f}')
print(f'Sensitivity:   {sensitivity:.4f}')
print(f'Specificity:   {specificity:.4f}')
print(f'Mean(Se, Sp):  {balanced_acc:.4f}')
print(f'{"="*40}')
print(f'\nConfusion Matrix:')
print(f'  TN={tn:6d}  FP={fp:6d}')
print(f'  FN={fn:6d}  TP={tp:6d}')

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['BG', 'Vein'], cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()
